# 🔬 SCOR-Py: Simulasi Monte Carlo & Analisis Hambatan (*Bottleneck*)

Notebook ini mendemonstrasikan bagaimana **SCOR-Py** mengubah metrik historis menjadi keputusan strategis melalui analisis skenario *What-If* berbasis simulasi Monte Carlo.

### Kasus Bisnis yang Dianalisis:
1. **Dampak Keterlambatan Pemasok** terhadap Service Level (OTIF).
2. **Optimasi Stok Pengaman (Safety Stock)** dan Pembebasan Modal Kerja.
3. **Estimasi Ketangkasan Rantai Pasok (Agility AG.1.1)**.


In [ ]:
import pandas as pd

from scorpy.simulator import SupplyChainSimulator

df_orders = pd.read_csv("../data/sample_orders.csv")
sim = SupplyChainSimulator.from_orders(df_orders, seed=42)
print("Simulator diinisialisasi dengan data transaksi.")

## 1. Skenario: Keterlambatan Pemasok (+3 Hari)

Bagaimana jika terjadi hambatan pengiriman dari vendor sebesar 3 hari? Berapa penurunan OTIF dan batas toleransi keyakinan 95%?


In [ ]:
result = sim.run(n_runs=1000, scenario={"supplier_lead_time_shift_days": 3})
otif_comparison = result.compare_to_baseline("OTIF_rate")

print(f"OTIF Baseline   : {otif_comparison['baseline'] * 100:.1f}%")
print(f"OTIF Proyeksi   : {otif_comparison['scenario_mean'] * 100:.1f}%")
print(
    f"Perubahan Rata2 : {otif_comparison['delta_mean'] * 100:+.1f}% ({otif_comparison['delta_pct']:+.1f}%)"
)
print(
    f"95% CI          : [{otif_comparison['ci_95_lower'] * 100:.1f}%, {otif_comparison['ci_95_upper'] * 100:.1f}%]"
)

### Ringkasan Lengkap Dampak ke Semua Metrik


In [ ]:
result.summary()

## 2. Skenario: Optimasi Safety Stock & Modal Kerja

Jika stok pengaman diturunkan 15%, berapa modal kerja yang dibebaskan dan bagaimana dampaknya terhadap siklus Cash-to-Cash?


In [ ]:
ss_sim = SupplyChainSimulator.simulate_safety_stock_adjustment(
    avg_inventory_value=2_100_000_000,
    annual_cogs=15_000_000_000,
    annual_revenue=22_500_000_000,
    accounts_receivable=1_800_000_000,
    accounts_payable=1_400_000_000,
    safety_stock_pct_of_inventory=0.40,
    safety_stock_change_pct=-0.15,
)

print(f"Modal Kerja Dibebaskan : Rp {ss_sim['working_capital_freed']:,.0f}")
print(
    f"Dampak ke IDS          : {ss_sim['delta_IDS_days']:+.1f} hari (dari {ss_sim['baseline_IDS_days']} ke {ss_sim['new_IDS_days']} hari)"
)
print(f"Dampak ke C2C          : {ss_sim['delta_C2C_days']:+.1f} hari")

## 3. Estimasi Ketangkasan (Agility AG.1.1: Upside Flexibility)

Berapa hari yang dibutuhkan perusahaan untuk memenuhi lonjakan permintaan +20% tanpa mengorbankan tingkat layanan?


In [ ]:
ag = sim.estimate_upside_flexibility(target_demand_increase_pct=0.20)
print(f"Metrik                     : {ag['metric']}")
print(f"Target Kenaikan Permintaan : +{ag['target_increase_pct']:.0f}%")
print(f"Waktu Adaptasi Estimasi    : {ag['estimated_days_to_adapt']} hari")